# Fire 2021 Interactive Prediction Notebook

This notebook does the following:
- Loads one fire from year 2021
- Uses **10 leading observations**
- Runs next-day model prediction
- Shows a day slider on a **Google Maps** basemap
- Accepts user-provided feature rasters and updates next-day prediction

**Constraint enforced**: no simulated or assumed raster generation.
User updates are only accepted from real raster inputs (GeoTIFF files).

In [9]:
import os
import sys
import base64
from io import BytesIO
from pathlib import Path
from dataclasses import replace

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import rasterio
from rasterio.enums import Resampling

import ipywidgets as widgets
from IPython.display import display, clear_output
from ipyleaflet import Map, TileLayer, ImageOverlay, LayersControl, WidgetControl

REPO_ROOT = Path('/u50/joshin10/WildfireSpreadPrediction')
BACKEND_SRC = REPO_ROOT / 'backend' / 'src'
WSTS_SRC = REPO_ROOT / 'src' / 'wsts' / 'src'

if str(BACKEND_SRC) not in sys.path:
    sys.path.insert(0, str(BACKEND_SRC))
if str(WSTS_SRC) not in sys.path:
    sys.path.insert(0, str(WSTS_SRC))

from wildfire_api.config import Settings
from wildfire_api.repository import WildfireRepository
from wildfire_api.preprocessing import SamplePreprocessor
from models import DomainAdversarialUTAELightning, ResNet18UTAELightning, UTAELightning, SMPModel

torch.set_grad_enabled(False)
print('Imports ready')

Imports ready


## Configuration
Set paths for your checkpoint and HDF5 root.

- `n_leading_observations` is fixed to **10** per your request.
- Fire year is fixed to **2021**.

In [10]:
CHECKPOINT_PATH = REPO_ROOT / 'src' / 'wsts' / 'lightning_logs'/ 'wildfire_progression'/ 'o6t4tkz7' / 'checkpoints'/ 'epoch=26-step=2295.ckpt'  # update if needed
HDF5_ROOT = Path('/u50/capstone/cs4zp6g17/data/hdf5')

settings = Settings(
    model_path=CHECKPOINT_PATH,
    hdf5_root=HDF5_ROOT,
    stats_years=(2018, 2019),
    default_year=2021,
    n_leading_observations=10,
    probability_threshold=0.5,
    flatten_temporal_dimension=False,
)

repo = WildfireRepository(settings)
preprocessor = SamplePreprocessor(settings)
catalog_2021 = repo.list_year(2021)
print(f'2021 fires available: {len(catalog_2021)}')
print('First 5 fire ids:', [m.fire_id for m in catalog_2021[:5]])

2021 fires available: 156
First 5 fire ids: ['fire_24935867', 'fire_24935874', 'fire_24935885', 'fire_24936357', 'fire_25017306']


In [11]:
MODEL_CLASS_MAP = {
    'DomainAdversarialUTAELightning': DomainAdversarialUTAELightning,
    'ResNet18UTAELightning': ResNet18UTAELightning,
    'UTAELightning': UTAELightning,
    'SMPModel': SMPModel,
}

MODEL_CLASS_NAME = 'DomainAdversarialUTAELightning'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ModelClass = MODEL_CLASS_MAP[MODEL_CLASS_NAME]
model = ModelClass.load_from_checkpoint(str(CHECKPOINT_PATH), map_location=device)
model.eval()
model.to(device)
print(f'Model loaded: {MODEL_CLASS_NAME} on {device}')

Model loaded: DomainAdversarialUTAELightning on cuda


## Load one fire (2021) and build 10-day timeline
The slider shows the active fire channel across the 10 leading observations.
Then we display the model's next-day prediction for that window.

In [12]:
def array_to_data_url(arr, cmap='hot', alpha=0.65):
    arr = np.asarray(arr, dtype=np.float32)
    if np.nanmax(arr) > np.nanmin(arr):
        norm = (arr - np.nanmin(arr)) / (np.nanmax(arr) - np.nanmin(arr) + 1e-8)
    else:
        norm = np.zeros_like(arr)

    cm = plt.get_cmap(cmap)
    rgba = cm(norm)
    rgba[..., 3] = alpha * (norm > 0).astype(np.float32)
    img = (rgba * 255).astype(np.uint8)

    pil_img = Image.fromarray(img)
    buf = BytesIO()
    pil_img.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    return f'data:image/png;base64,{b64}'

def bounds_from_center(lat, lon, height, width, meters_per_pixel=375):
    lat_deg_per_meter = 1 / 111320
    lon_deg_per_meter = 1 / (111320 * np.cos(np.deg2rad(lat)) + 1e-8)

    half_h_m = 0.5 * height * meters_per_pixel
    half_w_m = 0.5 * width * meters_per_pixel

    south = lat - half_h_m * lat_deg_per_meter
    north = lat + half_h_m * lat_deg_per_meter
    west = lon - half_w_m * lon_deg_per_meter
    east = lon + half_w_m * lon_deg_per_meter
    return ((south, west), (north, east))

# ---- Pick a significantly large fire from 2021 ----
# Heuristic size score: spatial footprint * number of prediction samples
size_rank = sorted(
    catalog_2021,
    key=lambda m: (m.height * m.width * max(m.samples, 1)),
    reverse=True,
 )

LARGE_FIRE_RANK = 0  # 0 = largest, 1 = second largest, ...
MANUAL_FIRE_ID = None  # e.g., 'fire_24935867' to override auto selection

if MANUAL_FIRE_ID:
    fire_id = MANUAL_FIRE_ID
else:
    fire_id = size_rank[LARGE_FIRE_RANK].fire_id

cube_obj = repo.load_cube(fire_id=fire_id, year=2021)
cube = cube_obj.cube  # (T, 23, H, W)
meta = cube_obj.metadata

if cube.shape[0] < 11:
    raise RuntimeError('Selected fire has fewer than 11 frames; cannot use 10 leading observations + next day label.')

sample_offset = min(0, cube.shape[0] - 11)
prep = preprocessor.prepare(cube, sample_offset=sample_offset)

x_input = prep.tensor.to(device)
with torch.inference_mode():
    y_pred = model(x_input)
    y_prob = torch.sigmoid(y_pred).detach().cpu().numpy()[0, 0]

# 10-day observed timeline from active fire raw channel (-1) before preprocessing
start = prep.sample_index
obs_10day = cube[start:start+10, -1]  # (10, H, W)

# Crop observed timeline to match model tensor spatial dims
crop_h, crop_w = prep.spatial_shape
orig_h, orig_w = obs_10day.shape[-2:]
h_off = (orig_h - crop_h) // 2
w_off = (orig_w - crop_w) // 2
obs_10day = obs_10day[:, h_off:h_off+crop_h, w_off:w_off+crop_w]

print(f'Fire: {fire_id} (2021)')
print(f'Approx size rank in 2021 catalog: {LARGE_FIRE_RANK + 1}/{len(size_rank)}')
print(f'Metadata -> HxW: {meta.height}x{meta.width}, time_steps: {meta.time_steps}, samples: {meta.samples}')
print(f'Cube shape: {cube.shape}, model input shape: {tuple(prep.tensor.shape)}')
print(f'10-day observed timeline shape: {obs_10day.shape}')
print(f'Next-day prediction shape: {y_prob.shape}')

Fire: fire_25294714 (2021)
Approx size rank in 2021 catalog: 1/156
Metadata -> HxW: 302x235, time_steps: 72, samples: 62
Cube shape: (72, 23, 302, 235), model input shape: (1, 10, 40, 288, 224)
10-day observed timeline shape: (10, 288, 224)
Next-day prediction shape: (288, 224)


In [13]:
google_sat = TileLayer(
    url='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attribution='Google',
    name='Google Satellite'
)

center = (meta.latitude, meta.longitude)
bounds = bounds_from_center(meta.latitude, meta.longitude, crop_h, crop_w, meters_per_pixel=375)

overlay_day = ImageOverlay(
    url=array_to_data_url(obs_10day[0], cmap='hot', alpha=0.7),
    bounds=bounds,
    name='Observed Active Fire (day slider)'
)

overlay_pred = ImageOverlay(
    url=array_to_data_url(y_prob, cmap='viridis', alpha=0.6),
    bounds=bounds,
    name='Predicted Next Day'
)

m = Map(center=center, zoom=10, basemap=google_sat)
m.add_layer(overlay_day)
m.add_layer(overlay_pred)
m.add_control(LayersControl(position='topright'))
m.fit_bounds(bounds)

day_slider = widgets.IntSlider(value=0, min=0, max=9, step=1, description='Day')
day_label = widgets.HTML(value='<b>Observed day:</b> 1 / 10')
zoom_slider = widgets.IntSlider(value=10, min=5, max=16, step=1, description='Zoom')

def on_day_change(change):
    if change['name'] == 'value':
        idx = change['new']
        overlay_day.url = array_to_data_url(obs_10day[idx], cmap='hot', alpha=0.7)
        day_label.value = f'<b>Observed day:</b> {idx+1} / 10'

def on_zoom_change(change):
    if change['name'] == 'value':
        m.zoom = change['new']

day_slider.observe(on_day_change, names='value')
zoom_slider.observe(on_zoom_change, names='value')

slider_box = widgets.VBox([day_label, day_slider, zoom_slider])
m.add_control(WidgetControl(widget=slider_box, position='bottomleft'))

display(m)

Map(center=[40.208848379959335, -121.05600540190937], controls=(ZoomControl(options=['position', 'zoom_in_text…

## User feature inputs (value-driven, generated rasters)
Provide scalar values (not GeoTIFF paths). The notebook deterministically generates raster layers for the latest timestep before next-day prediction.

Generated channels in the 23 raw channels:
- 0: VIIRS M11
- 1: VIIRS I2
- 3: NDVI
- 4: EVI2
- 5: Total precipitation
- 6: Wind speed (generated wind heatmap)
- 14: Elevation
- 12: Slope (optional)
- 13: Aspect (optional)

Wind heatmap generation uses:
- Base wind speed value (required)
- Turbulence intensity value (optional)
- Terrain modulation from existing slope/aspect rasters (deterministic)

Optionally, generated rasters can be exported as GeoTIFF files.

In [14]:
value_widgets = {
    'viirs_m11': widgets.FloatText(description='M11', value=2000.0, layout=widgets.Layout(width='48%')),
    'viirs_i2': widgets.FloatText(description='I2', value=3000.0, layout=widgets.Layout(width='48%')),
    'ndvi': widgets.FloatText(description='NDVI', value=0.35, layout=widgets.Layout(width='48%')),
    'evi2': widgets.FloatText(description='EVI2', value=0.25, layout=widgets.Layout(width='48%')),
    'precip': widgets.FloatText(description='Precip', value=2.0, layout=widgets.Layout(width='48%')),
    'elevation': widgets.FloatText(description='Elev', value=float(np.nanmean(cube[start:start+10, 14, h_off:h_off+crop_h, w_off:w_off+crop_w])), layout=widgets.Layout(width='48%')),
    'slope': widgets.FloatText(description='Slope*', value=float(np.nanmean(cube[start:start+10, 12, h_off:h_off+crop_h, w_off:w_off+crop_w])), layout=widgets.Layout(width='48%')),
    'aspect': widgets.FloatText(description='Aspect*', value=float(np.nanmean(cube[start:start+10, 13, h_off:h_off+crop_h, w_off:w_off+crop_w])), layout=widgets.Layout(width='48%')),
    'wind_speed': widgets.FloatText(description='Wind', value=6.0, layout=widgets.Layout(width='48%')),
    'turbulence': widgets.FloatText(description='Turb*', value=0.15, layout=widgets.Layout(width='48%')),
}

export_tif_checkbox = widgets.Checkbox(value=False, description='Export generated rasters to GeoTIFF')
export_dir_text = widgets.Text(
    description='Export dir',
    value=str(REPO_ROOT / 'featureAPI' / 'generated_rasters'),
    layout=widgets.Layout(width='95%')
)

rows = []
keys = list(value_widgets.keys())
for i in range(0, len(keys), 2):
    pair = [value_widgets[keys[i]]]
    if i + 1 < len(keys):
        pair.append(value_widgets[keys[i+1]])
    rows.append(widgets.HBox(pair))

submit_btn = widgets.Button(description='Generate rasters and update prediction', button_style='success')
status_out = widgets.Output()
heatmap_out = widgets.Output()

display(widgets.VBox(rows + [export_tif_checkbox, export_dir_text, submit_btn, status_out, heatmap_out]))

In [15]:
RAW_CHANNEL_MAP = {
    'viirs_m11': 0,
    'viirs_i2': 1,
    'ndvi': 3,
    'evi2': 4,
    'precip': 5,
    'wind_speed': 6,
    'elevation': 14,
    'slope': 12,
    'aspect': 13,
}

def deterministic_field(value, shape):
    return np.full(shape, float(value), dtype=np.float32)

def normalize01(arr):
    arr = np.asarray(arr, dtype=np.float32)
    mn, mx = np.nanmin(arr), np.nanmax(arr)
    if mx <= mn:
        return np.zeros_like(arr, dtype=np.float32)
    return (arr - mn) / (mx - mn + 1e-8)

def generate_wind_heatmap(base_wind, turbulence, slope_map, aspect_map):
    """
    Deterministic wind map from scalar controls + terrain effects.
    - slope increases wind on steeper areas
    - aspect introduces directional anisotropy
    - turbulence scales spatial variability deterministically from terrain gradients
    """
    slope_n = normalize01(slope_map)
    # aspect factor in [0,1] from cosine pattern
    aspect_factor = 0.5 * (1.0 + np.cos(np.deg2rad(aspect_map)))
    terrain_grad = np.hypot(*np.gradient(slope_n))
    terrain_grad_n = normalize01(terrain_grad)

    # deterministic composition
    wind = (
        float(base_wind)
        * (1.0 + 0.35 * slope_n)
        * (0.85 + 0.30 * aspect_factor)
        * (1.0 + float(turbulence) * terrain_grad_n)
    )
    return wind.astype(np.float32)

def write_generated_tif(path, arr, ref_lat, ref_lon, meters_per_pixel=375):
    from rasterio.transform import from_bounds
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    h, w = arr.shape
    b = bounds_from_center(ref_lat, ref_lon, h, w, meters_per_pixel=meters_per_pixel)
    south, west = b[0]
    north, east = b[1]
    transform = from_bounds(west, south, east, north, w, h)

    with rasterio.open(
        path,
        'w',
        driver='GTiff',
        height=h,
        width=w,
        count=1,
        dtype='float32',
        crs='EPSG:4326',
        transform=transform,
    ) as dst:
        dst.write(arr.astype(np.float32), 1)

def update_prediction_from_user(_):
    with status_out:
        clear_output()
        print('Generating deterministic raster inputs from scalar controls...')

    with heatmap_out:
        clear_output()

    raw_window = np.copy(cube[start:start+10])  # (10,23,H,W)
    raw_window = raw_window[:, :, h_off:h_off+crop_h, w_off:w_off+crop_w]

    # Existing terrain references from last observed timestep for deterministic modulation
    last_slope_ref = raw_window[-1, 12].astype(np.float32)
    last_aspect_ref = raw_window[-1, 13].astype(np.float32)

    try:
        generated = {}

        # Deterministic scalar -> raster for non-wind channels
        for key in ['viirs_m11', 'viirs_i2', 'ndvi', 'evi2', 'precip', 'elevation', 'slope', 'aspect']:
            val = float(value_widgets[key].value)
            generated[key] = deterministic_field(val, (crop_h, crop_w))

        # Wind heatmap from wind speed + turbulence + terrain
        base_wind = float(value_widgets['wind_speed'].value)
        turbulence = float(value_widgets['turbulence'].value)
        slope_for_wind = generated['slope'] if 'slope' in generated else last_slope_ref
        aspect_for_wind = generated['aspect'] if 'aspect' in generated else last_aspect_ref
        generated['wind_speed'] = generate_wind_heatmap(base_wind, turbulence, slope_for_wind, aspect_for_wind)

        # Write generated rasters into latest timestep channels
        for key, ch in RAW_CHANNEL_MAP.items():
            raw_window[-1, ch] = generated[key]

        # Build temporary cube with updated 10-day window
        temp_cube = np.copy(cube)
        temp_cube[start:start+10, :, h_off:h_off+crop_h, w_off:w_off+crop_w] = raw_window

        prep_user = preprocessor.prepare(temp_cube, sample_offset=sample_offset)
        x_user = prep_user.tensor.to(device)

        with torch.inference_mode():
            y_user = model(x_user)
            y_user_prob = torch.sigmoid(y_user).detach().cpu().numpy()[0, 0]

        overlay_pred.url = array_to_data_url(y_user_prob, cmap='viridis', alpha=0.65)

        exported_paths = []
        if export_tif_checkbox.value:
            export_dir = Path(export_dir_text.value).expanduser().resolve()
            export_dir.mkdir(parents=True, exist_ok=True)
            for key, arr in generated.items():
                out_path = export_dir / f'{fire_id}_{key}.tif'
                write_generated_tif(out_path, arr, meta.latitude, meta.longitude)
                exported_paths.append(str(out_path))

        with status_out:
            clear_output()
            print('Prediction updated from generated raster inputs.')
            print('Generated channels:', sorted(list(generated.keys())))
            print(f'Wind control -> base speed: {base_wind:.3f}, turbulence: {turbulence:.3f}')
            if exported_paths:
                print('Exported GeoTIFFs:')
                for p in exported_paths:
                    print(' -', p)

        with heatmap_out:
            keys = ['viirs_m11', 'viirs_i2', 'ndvi', 'evi2', 'precip', 'elevation', 'slope', 'aspect', 'wind_speed']
            n_plots = len(keys)
            cols = 3
            rows = int(np.ceil(n_plots / cols))
            fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
            axes = np.array(axes).reshape(-1)

            for idx, key in enumerate(keys):
                arr = generated[key]
                cmap = 'viridis' if key == 'wind_speed' else 'magma'
                im = axes[idx].imshow(arr, cmap=cmap)
                axes[idx].set_title(f'Generated raster: {key}')
                plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)

            for j in range(n_plots, len(axes)):
                axes[j].axis('off')

            plt.tight_layout()
            plt.show()

    except Exception as e:
        with status_out:
            clear_output()
            print('Update failed:')
            print(str(e))

submit_btn.on_click(update_prediction_from_user)
print('Submit handler ready.')

Submit handler ready.


In [16]:
# Rebind submit button to the current handler only (clears stale handlers from older notebook versions)
try:
    submit_btn._click_handlers.callbacks.clear()
except Exception:
    pass

submit_btn.on_click(update_prediction_from_user)
print('Submit button rebound to current value-based handler.')

Submit button rebound to current value-based handler.


## Notes
- Inputs are scalar values; rasters are generated deterministically per submit.
- Wind raster is computed from wind speed + turbulence and terrain modulation (slope/aspect).
- You can export generated feature rasters as GeoTIFF from the checkbox option.
- This allows value-based interaction while still producing rasterized model inputs each run.